In [6]:
import json
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

In [7]:
!pip install -U bitsandbytes>=0.46.1

In [8]:
print("GPU available:", torch.cuda.is_available())

GPU available: True


In [9]:
MODEL_ID    = "Qwen/Qwen2.5-1.5B-Instruct"   # small, ungated, easy to iterate
MAX_SEQ_LEN = 1024                            # chosen from the length percentiles (covers ~99%)

SYSTEM_MSG = (
    "You classify the intent behind the speaker's words. "
    "Choose the correct option(s). Reply with only a JSON object, no other text."
)

tok = AutoTokenizer.from_pretrained(MODEL_ID)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [10]:
data = load_dataset("yuweiyin/IntentGrasp", "all", split="train")

README.md:   0%|          | 0.00/8.41k [00:00<?, ?B/s]

all/train.parquet: reconstructing file:   0%|          |  0.00B / 44.5MB            

all/train.parquet: downloading bytes:           |  0.00B            

all/test.parquet: reconstructing file:   0%|          |  0.00B / 3.79MB            

all/test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/262759 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12909 [00:00<?, ? examples/s]

In [11]:
print(data['options'][0])

['To ask about flight distance.', 'To ask for ground service information.', 'To ask about flight time.', 'To ask for meal information.', 'To ask for airfare information.', 'To ask for the day of the week.', 'To ask for flight restrictions.', 'To book flights or ask for general flight information.', 'To ask for airline information.', 'To ask for the cheapest flight.']


In [12]:
def make_example(row):
    context  = row["context"]
    question = row["question"]

    # options serialized 0-based: the enumerate index IS the label
    options_block = "\n".join(
        f"({i}) {text}" for i, text in enumerate(row["options"])
    )

    # answer = index strings (str(i), always a list -> handles multi-answer)
    answer = [str(i) for i in row["answer_index"]]
    # intent = option text looked up BY INDEX (safer than copying answer_intent)
    intent = [row["options"][i] for i in row["answer_index"]]

    user_str = (
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        f"Options:\n{options_block}"
    )

    completion = json.dumps({"answer": answer, "intent": intent})

    messages = [
        {"role": "system",    "content": SYSTEM_MSG},
        {"role": "user",      "content": user_str},
        {"role": "assistant", "content": completion},
    ]
    return {"messages": messages, "completion": completion}


In [13]:
def is_valid(row):
    n = len(row["options"])
    return all(0 <= i < n for i in row["answer_index"])

clean = data.filter(is_valid)
print("before:", len(data), " after:", len(clean), " dropped:", len(data) - len(clean))

Filter:   0%|          | 0/262759 [00:00<?, ? examples/s]

before: 262759  after: 262420  dropped: 339


In [14]:
ex = make_example(clean[0])                    # one formatted example
text = tok.apply_chat_template(
    ex["messages"],                            # all 3 turns: system, user, assistant
    tokenize=False,                            # readable string, not token IDs
    add_generation_prompt=False,               # training: keep the answer in
)
print(text)

<|im_start|>system
You classify the intent behind the speaker's words. Choose the correct option(s). Reply with only a JSON object, no other text.<|im_end|>
<|im_start|>user
Context:
user: i want to fly from boston at 838 am and arrive in denver at 1110 in the morning

Question:
What is the intent of the user?

Options:
(0) To ask about flight distance.
(1) To ask for ground service information.
(2) To ask about flight time.
(3) To ask for meal information.
(4) To ask for airfare information.
(5) To ask for the day of the week.
(6) To ask for flight restrictions.
(7) To book flights or ask for general flight information.
(8) To ask for airline information.
(9) To ask for the cheapest flight.<|im_end|>
<|im_start|>assistant
{"answer": ["7"], "intent": ["To book flights or ask for general flight information."]}<|im_end|>



In [15]:
split    = clean.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]        # model learns from this
val_ds   = split["test"]         # VALIDATION (watch overfitting) - NOT the official test set
print("train:", len(train_ds), " val:", len(val_ds))

train: 249299  val: 13121


In [16]:
train_formatted = train_ds.map(make_example)   # every row now has "messages" + "completion"
val_formatted   = val_ds.map(make_example)     # same for validation
print(train_formatted[0].keys())               # confirm "messages" is now present

Map:   0%|          | 0/249299 [00:00<?, ? examples/s]

Map:   0%|          | 0/13121 [00:00<?, ? examples/s]

dict_keys(['id', 'metadata', 'speaker', 'context', 'question', 'options', 'answer_intent', 'answer_index', 'messages', 'completion'])


In [17]:
import torch
print("GPU:", torch.cuda.is_available())

GPU: True


In [18]:
!pip install -q -U trl peft bitsandbytes accelerate

In [19]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

In [20]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # store frozen weights in 4-bit
    bnb_4bit_quant_type="nf4",              # the 4-bit format made for NN weights
    bnb_4bit_compute_dtype=torch.bfloat16,  # math runs in higher precision
    bnb_4bit_use_double_quant=True,         # squeeze a little more memory
)

In [21]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",                      # place on GPU
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [22]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

In [23]:
model = prepare_model_for_kbit_training(model)

In [24]:
lora_config = LoraConfig(
    r=16,                     # rank — the "width" of the skinny strips
    lora_alpha=32,            # scaling for the adapter's contribution
    target_modules="all-linear",   # which layers get an adapter
    lora_dropout=0.05,        # light regularization
    bias="none",
    task_type="CAUSAL_LM",    # this is a text-generation model
)

In [25]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [26]:
from trl import SFTTrainer

In [27]:
from trl import SFTConfig

args = SFTConfig(
    output_dir="intentgrasp-qlora",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=1,
    max_length=1024,
    logging_steps=25,
    bf16=True,
    report_to="none",
)

In [28]:
def to_text(row):
    text = tok.apply_chat_template(
        row["messages"],
        tokenize=False,
        add_generation_prompt=False,   # training: full conversation, answer included
    )
    return {"text": text}

In [29]:
small_train = train_formatted.select(range(500)).map(to_text)
small_val   = val_formatted.select(range(100)).map(to_text)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=small_train,
    eval_dataset=small_val,
    processing_class=tok,
)
trainer.train()

Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss


Step,Training Loss
25,1.557894


TrainOutput(global_step=32, training_loss=1.5374843180179596, metrics={'train_runtime': 1159.4675, 'train_samples_per_second': 0.431, 'train_steps_per_second': 0.028, 'total_flos': 1913990002704384.0, 'train_loss': 1.5374843180179596, 'entropy': 1.4460296355761015, 'num_tokens': 118575.0, 'mean_token_accuracy': 0.7093364688066336, 'epoch': 1.0})

In [30]:
from google.colab import drive
drive.mount('/content/drive')
train_formatted.save_to_disk("/content/drive/MyDrive/intentgrasp_train_formatted")
val_formatted.save_to_disk("/content/drive/MyDrive/intentgrasp_val_formatted")

Mounted at /content/drive


Saving the dataset (0/1 shards):   0%|          | 0/249299 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/13121 [00:00<?, ? examples/s]

In [31]:
import inspect
print(list(inspect.signature(SFTConfig.__init__).parameters))

['self', 'output_dir', 'per_device_train_batch_size', 'num_train_epochs', 'max_steps', 'learning_rate', 'lr_scheduler_type', 'lr_scheduler_kwargs', 'warmup_steps', 'optim', 'optim_args', 'weight_decay', 'adam_beta1', 'adam_beta2', 'adam_epsilon', 'optim_target_modules', 'gradient_accumulation_steps', 'average_tokens_across_devices', 'max_grad_norm', 'label_smoothing_factor', 'bf16', 'fp16', 'bf16_full_eval', 'fp16_full_eval', 'tf32', 'gradient_checkpointing', 'gradient_checkpointing_kwargs', 'torch_compile', 'torch_compile_backend', 'torch_compile_mode', 'use_liger_kernel', 'liger_kernel_config', 'use_cache', 'neftune_noise_alpha', 'torch_empty_cache_steps', 'auto_find_batch_size', 'logging_strategy', 'logging_steps', 'logging_first_step', 'log_on_each_node', 'logging_nan_inf_filter', 'include_num_input_tokens_seen', 'log_level', 'log_level_replica', 'disable_tqdm', 'report_to', 'run_name', 'project', 'trackio_space_id', 'trackio_bucket_id', 'trackio_static_space_id', 'eval_strategy', 

In [43]:
from trl import SFTConfig

args = SFTConfig(
    output_dir="intentgrasp-qlora",
    per_device_train_batch_size=32,          # A100; L4 → 8-16
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    num_train_epochs=1,
    lr_scheduler_type="cosine",
    warmup_steps=100,                        # a bit more warmup for a long run
    max_length=1024,
    assistant_only_loss=True,
    gradient_checkpointing=True,             # ← memory safety for the long run
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,                          # ← eval less often (full run has ~7800 steps)
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    report_to="none",
    seed=42,
)

In [47]:
train_full = train_formatted.remove_columns(
    [c for c in train_formatted.column_names if c != "messages"])
val_eval = val_formatted.select(range(1000)).remove_columns(
    [c for c in val_formatted.column_names if c != "messages"])

In [48]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_full,
    eval_dataset=val_eval,
    processing_class=tok,
)

Tokenizing train dataset:   0%|          | 0/249299 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/249299 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/249299 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/249299 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [49]:
trainer.train()
trainer.save_model("intentgrasp_adapter_full")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
500,0.023080,0.021447,0.023901,3727050.000000,0.992826
1000,0.017309,0.018955,0.013475,7477091.000000,0.993657
1500,0.015289,0.014679,0.015544,11192295.000000,0.994902
2000,0.014463,0.012929,0.010669,14926626.000000,0.995694
2500,0.012652,0.012022,0.012185,18684517.000000,0.995865
3000,0.012878,0.012134,0.015926,22434716.000000,0.995754
3500,0.009683,0.011213,0.009741,26151570.000000,0.996029
4000,0.009772,0.010133,0.009299,29866616.000000,0.996546
4500,0.008647,0.010088,0.008892,33589453.000000,0.996193
5000,0.009079,0.009772,0.009775,37328896.000000,0.996536


KeyboardInterrupt: 

In [50]:
trainer.save_model("intentgrasp_adapter_full")

import shutil
shutil.make_archive("intentgrasp_adapter_full", "zip", "intentgrasp_adapter_full")
from google.colab import files
files.download("intentgrasp_adapter_full.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [51]:
print(type(model))

<class 'peft.peft_model.PeftModelForCausalLM'>


In [52]:
import json, torch

def score_one(row, model):
    ex = make_example(row)
    prompt = tok.apply_chat_template(
        ex["messages"][:2],              # system + user only — withhold answer
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,             # greedy = deterministic
            pad_token_id=tok.eos_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[1]:]        # keep only NEW tokens
    text = tok.decode(gen, skip_special_tokens=True)

    true_set = set(json.loads(ex["completion"])["answer"])
    try:
        pred_set = set(json.loads(text)["answer"])
        return True, (pred_set == true_set), text       # valid, correct, raw
    except Exception:
        return False, False, text                       # invalid JSON

In [54]:
model.config.use_cache = True                 # re-enable the KV cache for generation
model.gradient_checkpointing_disable()        # turn off training-only memory trick
model.eval()                                   # inference mode

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [55]:
v, c, text = score_one(val_ds[0], model)
print("model output:", text)
print("valid:", v, "correct:", c)

model output: {"answer": ["6"], "intent": ["To inquire about the wrong exchange rate for cash withdrawal."]}
valid: True correct: True


In [57]:
def evaluate(model, n=200):
    valid = correct = 0
    for row in val_ds.select(range(n)):
        v, c, _ = score_one(row, model)
        valid += v
        correct += c
    return valid/n, correct/n

ft_valid, ft_acc = evaluate(model)
print(f"FINETUNED — validity: {ft_valid:.1%}, accuracy: {ft_acc:.1%}")

FINETUNED — validity: 100.0%, accuracy: 90.5%


In [62]:
from transformers import AutoModelForCausalLM
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.use_cache = True
base_model.eval()

base_valid, base_acc = evaluate(base_model)
print(f"BASE — validity: {base_valid:.1%}, accuracy: {base_acc:.1%}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

BASE — validity: 0.0%, accuracy: 0.0%


In [64]:
import json, re

def score_one_lenient(row, model):
    ex = make_example(row)
    prompt = tok.apply_chat_template(ex["messages"][:2], tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    true_set = set(json.loads(ex["completion"])["answer"])

    # lenient: pull ANY integer(s) out of whatever the model said
    nums = re.findall(r'\d+', text)
    pred_set = set(nums[:len(true_set)])   # take as many numbers as there are true answers
    strict_valid = False
    try:
        obj = json.loads(text)
        strict_valid = "answer" in obj      # did it match OUR schema?
    except Exception:
        pass
    return strict_valid, (pred_set == true_set), text

In [65]:
def evaluate_lenient(model, n=200):
    strict_ok = correct = 0
    for row in val_ds.select(range(n)):
        s, c, _ = score_one_lenient(row, model)
        strict_ok += s
        correct += c
    return strict_ok/n, correct/n

base_strict, base_acc_lenient = evaluate_lenient(base_model)
print(f"BASE — matches our schema: {base_strict:.1%}, lenient accuracy: {base_acc_lenient:.1%}")

BASE — matches our schema: 0.5%, lenient accuracy: 41.5%


In [66]:
for row in val_ds.select(range(15)):
    v, c, text = score_one(row, model)
    true = json.loads(make_example(row)["completion"])["answer"]
    print(f"correct={c} | true={true} | model={text[:80]}")

correct=True | true=['6'] | model={"answer": ["6"], "intent": ["To inquire about the wrong exchange rate for cash 
correct=False | true=['7'] | model={"answer": ["7", "8"], "intent": ["To call now or schedule a call.", "To ask for
correct=True | true=['8'] | model={"answer": ["8"], "intent": ["To inquire about getting the PIN."]}
correct=True | true=['5'] | model={"answer": ["5"], "intent": ["To encourage the boy."]}
correct=True | true=['4'] | model={"answer": ["4"], "intent": ["To set the reminder."]}
correct=True | true=['1'] | model={"answer": ["1"], "intent": ["To add music to the playlist."]}
correct=True | true=['5'] | model={"answer": ["5"], "intent": ["To inquire about the declined card payment."]}
correct=True | true=['0'] | model={"answer": ["0"], "intent": ["To add music to the playlist."]}
correct=True | true=['0'] | model={"answer": ["0"], "intent": ["To ask for career advice."]}
correct=True | true=['5'] | model={"answer": ["5"], "intent": ["To find the taxi."]}
correct=

In [67]:
import json, re

def get_prediction(row, model):
    ex = make_example(row)
    prompt = tok.apply_chat_template(ex["messages"][:2], tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text

In [68]:
import pandas as pd

rows_data = []
for row in val_ds.select(range(20)):
    ex = make_example(row)
    true = json.loads(ex["completion"])["answer"]

    ft_text   = get_prediction(row, model)        # your finetuned model
    base_text = get_prediction(row, base_model)   # untrained base

    rows_data.append({
        "context": row["context"][:60],           # truncate long contexts for display
        "true_answer": true,
        "finetuned_output": ft_text[:70],
        "base_output": base_text[:70],
    })

df = pd.DataFrame(rows_data)
df   # last line displays as a table in the notebook

,context,true_answer,finetuned_output,base_output
0,costumer: The exchange rate for foreign ATM cu...,[6],"{""answer"": [""6""], ""intent"": [""To inquire about...","{""intent"": 4}"
1,user: thank you please contact me date1 date2 ...,[7],"{""answer"": [""7"", ""8""], ""intent"": [""To call now...","{""intent"": (3)}"
2,costumer: The card PIN is not visible anywhere?,[8],"{""answer"": [""8""], ""intent"": [""To inquire about...","{""intent"": 8}"
3,### Situation: Ted is walking down a street an...,[5],"{""answer"": [""5""], ""intent"": [""To encourage the...","```json\n{""intent"": (5)}\n```"
4,user: remind me to shop for running shoes,[4],"{""answer"": [""4""], ""intent"": [""To set the remin...","{""intent"": ""To set the reminder.""}"
5,user: Add an artist to my playlist Domingo Indie,[1],"{""answer"": [""1""], ""intent"": [""To add music to ...","```json\n{\n ""intent"": (1)\n}\n```"
6,costumer: My card payment wasn't declined,[5],"{""answer"": [""5""], ""intent"": [""To inquire about...","```json\n{""intent"": (5)}\n```"
7,user: Add circus to my Post Garage Wave Reviv...,[0],"{""answer"": [""0""], ""intent"": [""To add music to ...","{""intent"": (0)}"
8,user: Can you help me to find an occupation,[0],"{""answer"": [""0""], ""intent"": [""To ask for caree...","{""intent"": ""To ask for career advice.""}"
9,### Dialogue:\nuser: Hello! I'm looking for a ...,[5],"{""answer"": [""5""], ""intent"": [""To find the taxi...","{""intent"": ""To find tourist attraction.""}"


In [69]:
df.to_csv("comparison_base_vs_finetuned.csv", index=False)
from google.colab import files
files.download("comparison_base_vs_finetuned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [70]:
print(df.to_markdown(index=False))

| context                                                      | true_answer   | finetuned_output                                                       | base_output                                                            |
|:-------------------------------------------------------------|:--------------|:-----------------------------------------------------------------------|:-----------------------------------------------------------------------|
| costumer: The exchange rate for foreign ATM currency is wron | ['6']         | {"answer": ["6"], "intent": ["To inquire about the wrong exchange rate | {"intent": 4}                                                          |
| user: thank you please contact me date1 date2 via text works | ['7']         | {"answer": ["7", "8"], "intent": ["To call now or schedule a call.", " | {"intent": (3)}                                                        |
| costumer: The card PIN is not visible anywhere?              | ['8']         | {"answer": 

In [72]:
gem = load_dataset("yuweiyin/IntentGrasp", "gem", split="test")
gem_clean = gem.filter(is_valid)
print("gem:", len(gem), "→ clean:", len(gem_clean))

gem/test.parquet: reconstructing file:   0%|          |  0.00B /  190kB            

gem/test.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/470 [00:00<?, ? examples/s]

Filter:   0%|          | 0/470 [00:00<?, ? examples/s]

gem: 470 → clean: 470


In [75]:
def evaluate(model, dataset, n=200):
    valid = correct = 0
    rows = dataset.select(range(min(n, len(dataset))))
    for row in rows:
        v, c, _ = score_one(row, model)
        valid += v
        correct += c
    return valid/len(rows), correct/len(rows)

In [76]:
gem_valid, gem_acc = evaluate(model, gem_clean, n=len(gem_clean))
print(f"GEM — validity: {gem_valid:.1%}, accuracy: {gem_acc:.1%}")

GEM — validity: 100.0%, accuracy: 37.9%
